# NOWCASTING VAB MANABÍ 2026

## Dashboard de Predicción en Tiempo Real

### Demostración de predicción de alta frecuencia con análisis de sentimiento

---

- Sección 1: Resumen de Validación
- Sección 2: Nowcasting 2026

**Objetivo:** Demostrar capacidad de nowcasting mensual adaptable

In [ ]:
# Setup
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
from sklearn.linear_model import Ridge
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')
path_proyecto = "/content/drive/MyDrive/TitulacionF"

print("NOWCASTING VAB MANABÍ 2026")
print("="*70)
print(f"Fecha: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
print("\nLibrerías cargadas correctamente")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOWCASTING VAB MANABÍ 2026
Fecha: 17/03/2026 04:23

Librerías cargadas correctamente


In [ ]:
# Cargar datos
print("\nCargando datos...")

try:
    df_sentimiento = pd.read_csv(f"{path_proyecto}/sentimiento_mensual_2020_2026.csv")
except:
    df_sentimiento = pd.read_csv(f"{path_proyecto}/sentimiento_mensual_2020_2024.csv")

print(f"Datos cargados: {len(df_sentimiento)} filas")
print(f"Años: {sorted(df_sentimiento['año'].unique())}")

# Cargar VAB histórico
df_vab = pd.read_csv(f"{path_proyecto}/vab_manabi_2020_2023.csv")

# VAB conocidos (oficiales BCE)
VAB_2024_REAL = 5768.40
VAB_2025_REAL = 5956.70

print("\nVAB oficiales (BCE):")
print(f"  2024: ${VAB_2024_REAL:,.2f}M")
print(f"  2025: ${VAB_2025_REAL:,.2f}M")


Cargando datos...
Datos cargados: 75 filas
Años: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

VAB oficiales (BCE):
  2024: $5,768.40M
  2025: $5,956.70M


---

# SECCIÓN 1: RESUMEN DE VALIDACIÓN

## Resultados en años anteriores (2024-2025)

---

In [ ]:
# Resultados validados (desde NB3 y NB4)
print("\n" + "="*70)
print("RESUMEN DE VALIDACIÓN")
print("="*70)

resultados = {
    'Año': ['2024', '2025', 'Promedio'],
    'VAB Real (M USD)': [5768.40, 5956.70, 5862.55],
    'Predicción (M USD)': [5745.17, 6059.45, 5902.31],
    'Error (%)': [0.40, 1.72, 1.06],
    'ARIMA Error (%)': [2.01, 3.03, 2.52],
    'Mejora vs ARIMA (%)': [79.9, 43.0, 61.5]
}

df_resultados = pd.DataFrame(resultados)

print("\nResultados de validación:")
print(df_resultados.to_string(index=False))

print("\nConclusiones principales:")
print("  - Error promedio: 1.06% (muy bajo)")
print("  - Mejora sobre ARIMA: 61.5%")
print("  - Modelo validado en 2 años consecutivos")
print("  - Listo para nowcasting 2026")


RESUMEN DE VALIDACIÓN

Resultados de validación:
     Año  VAB Real (M USD)  Predicción (M USD)  Error (%)  ARIMA Error (%)  Mejora vs ARIMA (%)
    2024           5768.40             5745.17       0.40             2.01                 79.9
    2025           5956.70             6059.45       1.72             3.03                 43.0
Promedio           5862.55             5902.31       1.06             2.52                 61.5

Conclusiones principales:
  - Error promedio: 1.06% (muy bajo)
  - Mejora sobre ARIMA: 61.5%
  - Modelo validado en 2 años consecutivos
  - Listo para nowcasting 2026


---

# SECCIÓN 2: NOWCASTING 2026 EN TIEMPO REAL

## Predicción adaptable con datos disponibles

---

In [ ]:
# Detectar datos 2026
df_2026 = df_sentimiento[df_sentimiento['año'] == 2026].copy()
meses_disponibles = sorted(df_2026['mes'].unique()) if len(df_2026) > 0 else []

print("\n" + "="*70)
print("NOWCASTING VAB 2026")
print("="*70)

if len(meses_disponibles) == 0:
    print("\nNo hay datos de 2026 disponibles")
    print("\nPara generar nowcasting:")
    print("  1. Ejecutar scraping de noticias 2026")
    print("  2. Procesar con análisis de sentimiento")
    print("  3. Re-ejecutar este notebook")
else:
    print(f"\nMeses detectados: {len(meses_disponibles)}/12")
    print(f"Cobertura: {len(meses_disponibles)/12*100:.0f}%")
    print(f"Último mes: {max(meses_disponibles)}/2026")

    # Nivel de confianza según meses
    if len(meses_disponibles) <= 2:
        confianza = "Muy Baja"
        incertidumbre = 0.15
    elif len(meses_disponibles) <= 4:
        confianza = "Baja"
        incertidumbre = 0.12
    elif len(meses_disponibles) <= 6:
        confianza = "Media"
        incertidumbre = 0.08
    elif len(meses_disponibles) <= 9:
        confianza = "Alta"
        incertidumbre = 0.05
    else:
        confianza = "Muy Alta"
        incertidumbre = 0.03

    print(f"Nivel de confianza: {confianza}")
    print(f"Incertidumbre: ±{incertidumbre*100:.0f}%")


NOWCASTING VAB 2026

Meses detectados: 3/12
Cobertura: 25%
Último mes: 3/2026
Nivel de confianza: Baja
Incertidumbre: ±12%


In [ ]:
# Entrenar modelo
if len(meses_disponibles) > 0:
    print("\nEntrenando modelo Ridge...")

    # Preparar datos train (2020-2025)
    df_train = df_sentimiento[df_sentimiento['año'].between(2020, 2025)].copy()
    df_train = df_train.sort_values('año_mes').reset_index(drop=True)

    # Calcular VAB mensual aproximado
    vab_dict = {
        2020: df_vab[df_vab['año']==2020]['vab_corriente'].values[0],
        2021: df_vab[df_vab['año']==2021]['vab_corriente'].values[0],
        2022: df_vab[df_vab['año']==2022]['vab_corriente'].values[0],
        2023: df_vab[df_vab['año']==2023]['vab_corriente'].values[0],
        2024: VAB_2024_REAL,
        2025: VAB_2025_REAL
    }

    df_train['vab_mensual'] = df_train['año'].map(vab_dict) / 12
    df_train['vab_variacion_pct'] = df_train['vab_mensual'].pct_change() * 100
    df_train['vab_mes_anterior'] = df_train['vab_mensual'].shift(1)
    df_train['vab_promedio_3m'] = df_train['vab_mensual'].rolling(3, min_periods=1).mean()

    # Features
    feature_cols = ['sentimiento_promedio', 'pct_positivas', 'pct_negativas',
                   'volatilidad_movil_3m', 'num_noticias',
                   'vab_mes_anterior', 'vab_promedio_3m']

    df_train_clean = df_train.dropna(subset=['vab_variacion_pct'] + feature_cols)

    X_train = df_train_clean[feature_cols].values
    y_train = df_train_clean['vab_variacion_pct'].values

    # Entrenar
    modelo = Ridge(alpha=0.1)
    modelo.fit(X_train, y_train)

    # Rangos para normalización
    rangos = {col: {'max': df_train_clean[col].max(),
                    'mean': df_train_clean[col].mean()}
              for col in feature_cols}

    print(f"Modelo entrenado: {len(y_train)} observaciones")
    print(f"num_noticias - Max train: {rangos['num_noticias']['max']:.0f}")
    print(f"num_noticias - Mean train: {rangos['num_noticias']['mean']:.0f}")


Entrenando modelo Ridge...
Modelo entrenado: 71 observaciones
num_noticias - Max train: 568
num_noticias - Mean train: 151


In [ ]:
# Predecir 2026 mes por mes
if len(meses_disponibles) > 0:
    print("\nPrediciendo VAB 2026 mes por mes...")
    print("="*70)

    vab_actual = VAB_2025_REAL / 12
    vab_promedio_3m = vab_actual
    predicciones = []

    for mes in meses_disponibles:
        row = df_2026[df_2026['mes'] == mes].iloc[0]

        # Features
        features = {
            'sentimiento_promedio': row['sentimiento_promedio'],
            'pct_positivas': row['pct_positivas'],
            'pct_negativas': row['pct_negativas'],
            'volatilidad_movil_3m': row['volatilidad_movil_3m'],
            'num_noticias': row['num_noticias'],
            'vab_mes_anterior': vab_actual,
            'vab_promedio_3m': vab_promedio_3m
        }

        # Normalización 0.75 (igual que NB4)
        num_original = features['num_noticias']
        if num_original > rangos['num_noticias']['max']:
            ratio = num_original / rangos['num_noticias']['mean']
            features['num_noticias'] = rangos['num_noticias']['mean'] * (ratio ** 0.75)
            print(f"Mes {mes}: num_noticias {num_original:.0f} → {features['num_noticias']:.0f} (normalizado)")

        # Predecir
        X_mes = np.array([[features[col] for col in feature_cols]])
        variacion = modelo.predict(X_mes)[0]
        variacion = np.clip(variacion, -10, 10)
        print(f"Mes {mes}: var={variacion:+.2f}%, sent={features['sentimiento_promedio']:.3f}, num_not={features['num_noticias']:.0f}")

        vab_mes = vab_actual * (1 + variacion/100)

        predicciones.append({
            'mes': mes,
            'vab': vab_mes,
            'variacion': variacion,
            'sentimiento': row['sentimiento_promedio'],
            'num_noticias': num_original
        })

        # Actualizar
        vab_actual = vab_mes
        if len(predicciones) >= 3:
            vab_promedio_3m = np.mean([p['vab'] for p in predicciones[-3:]])

    df_pred = pd.DataFrame(predicciones)

    # Proyección anual
    vab_parcial = df_pred['vab'].sum()
    vab_anual_proyectado = vab_parcial * (12 / len(meses_disponibles))

    print("\n" + "="*70)
    print("PROYECCIÓN VAB 2026")
    print("="*70)
    print(f"\nVAB parcial ({len(meses_disponibles)} meses): ${vab_parcial:,.2f}M")
    print(f"Proyección anual: ${vab_anual_proyectado:,.2f}M")
    print(f"Rango: ${vab_anual_proyectado*(1-incertidumbre):,.2f}M - ${vab_anual_proyectado*(1+incertidumbre):,.2f}M")
    print(f"\nVariación vs 2025: {((vab_anual_proyectado/VAB_2025_REAL)-1)*100:+.2f}%")
    print(f"Confianza: {confianza}")



Prediciendo VAB 2026 mes por mes...
Mes 1: var=-0.05%, sent=0.193, num_not=544
Mes 2: var=+0.06%, sent=0.341, num_not=449
Mes 3: var=-0.01%, sent=0.233, num_not=433

PROYECCIÓN VAB 2026

VAB parcial (3 meses): $1,488.97M
Proyección anual: $5,955.86M
Rango: $5,241.16M - $6,670.57M

Variación vs 2025: -0.01%
Confianza: Baja


In [ ]:
# Gráfico mejorado
if len(meses_disponibles) > 0:
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # VAB mensual
    fig.add_trace(
        go.Scatter(
            x=df_pred['mes'],
            y=df_pred['vab'],
            mode='lines+markers',
            name='VAB Mensual Predicho',
            line=dict(color='rgb(44, 160, 44)', width=3),
            marker=dict(size=10),
            hovertemplate='Mes %{x}<br>VAB: $%{y:.2f}M<extra></extra>'
        ),
        secondary_y=False
    )

    # Sentimiento (eje derecho)
    fig.add_trace(
        go.Scatter(
            x=df_pred['mes'],
            y=df_pred['sentimiento'],
            mode='lines+markers',
            name='Sentimiento',
            line=dict(color='rgb(31, 119, 180)', width=2),
            marker=dict(size=6),
            hovertemplate='Mes %{x}<br>Sentimiento: %{y:.3f}<extra></extra>'
        ),
        secondary_y=True
    )

    # Banda de confianza
    vab_upper = df_pred['vab'] * (1 + incertidumbre)
    vab_lower = df_pred['vab'] * (1 - incertidumbre)

    fig.add_trace(
        go.Scatter(
            x=list(df_pred['mes']) + list(df_pred['mes'][::-1]),
            y=list(vab_upper) + list(vab_lower[::-1]),
            fill='toself',
            fillcolor='rgba(44, 160, 44, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            name=f'Rango ±{incertidumbre*100:.0f}%',
            showlegend=True,
            hoverinfo='skip'
        ),
        secondary_y=False
    )

    # Línea de sentimiento neutral
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray",
        annotation_text="Sentimiento neutral",
        secondary_y=True
    )

    # Layout
    fig.update_layout(
        title=f'Nowcasting VAB Manabí 2026 - {len(meses_disponibles)} meses disponibles<br><sub>Actualizado: {datetime.now().strftime("%d %b %Y")}</sub>',
        xaxis_title='Mes',
        yaxis_title='VAB Mensual (M USD)',
        yaxis2_title='Sentimiento Promedio',
        height=550,
        hovermode='x unified',
        template='plotly_white'
    )

    fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)

    fig.show()

    # Tabla de datos
    print("\n" + "="*70)
    print("SENTIMIENTO DETECTADO")
    print("="*70)
    print("\n" + df_pred[['mes', 'sentimiento', 'num_noticias', 'vab']].to_string(index=False))


SENTIMIENTO DETECTADO

 mes  sentimiento  num_noticias        vab
   1     0.193015           544 496.142609
   2     0.340757           449 496.426628
   3     0.233256           433 496.396290


In [ ]:
# Tabla comparativa
if len(meses_disponibles) > 0:
    print("\n" + "="*70)
    print("COMPARACIÓN HISTÓRICA")
    print("="*70)

    comparacion = pd.DataFrame({
        'Métrica': [
            'VAB Anual',
            'Sentimiento Promedio',
            'Noticias Totales',
            'Meses Disponibles'
        ],
        '2024': [
            f'${VAB_2024_REAL:,.2f}M',
            f"{df_sentimiento[df_sentimiento['año']==2024]['sentimiento_promedio'].mean():.3f}",
            f"{int(df_sentimiento[df_sentimiento['año']==2024]['num_noticias'].sum())}",
            '12'
        ],
        '2025': [
            f'${VAB_2025_REAL:,.2f}M',
            f"{df_sentimiento[df_sentimiento['año']==2025]['sentimiento_promedio'].mean():.3f}",
            f"{int(df_sentimiento[df_sentimiento['año']==2025]['num_noticias'].sum())}",
            '12'
        ],
        '2026': [
            f'${vab_anual_proyectado:,.2f}M (proyección)',
            f"{df_pred['sentimiento'].mean():.3f}",
            f"{int(df_pred['num_noticias'].sum())}",
            f'{len(meses_disponibles)}'
        ]
    })

    print("\n" + comparacion.to_string(index=False))


COMPARACIÓN HISTÓRICA

             Métrica       2024       2025                    2026
           VAB Anual $5,768.40M $5,956.70M $5,955.86M (proyección)
Sentimiento Promedio      0.284      0.272                   0.256
    Noticias Totales       2193       4831                    1426
   Meses Disponibles         12         12                       3


---

# SECCIÓN 3: METODOLOGÍA

---

In [ ]:
print("\n" + "="*70)
print("METODOLOGÍA DEL NOWCASTING")
print("="*70)

print("""
MODELO:
  Ridge Regression (alpha=0.1)
  Target: Variación % mensual del VAB
  Features: 7 variables (sentimiento + inercia económica)

ENTRENAMIENTO:
  Datos: 2020-2025 (72 meses)
  Validación: 2024, 2025 (24 meses)
  Error promedio: 1.06%

NORMALIZACIÓN:
  num_noticias → potencia 0.75 si excede máximo histórico
  Razón: Explosión de cobertura mediática 2024-2025

NOWCASTING 2026:
  Método: Predicción mes por mes con mismo modelo validado
  Adaptable: Funciona con 1-12 meses disponibles
  Actualización: Automática al ejecutar notebook

NIVEL DE CONFIANZA:
  1-2 meses:   Muy Baja (±15%)
  3-4 meses:   Baja (±12%)
  5-6 meses:   Media (±8%)
  7-9 meses:   Alta (±5%)
  10-12 meses: Muy Alta (±3%)

VENTAJA PRINCIPAL:
  Frecuencia mensual vs anual del BCE
  Alerta temprana de cambios económicos
  Basado en señales actuales, no solo historia
""")

print("="*70)



METODOLOGÍA DEL NOWCASTING

MODELO:
  Ridge Regression (alpha=0.1)
  Target: Variación % mensual del VAB
  Features: 7 variables (sentimiento + inercia económica)

ENTRENAMIENTO:
  Datos: 2020-2025 (72 meses)
  Validación: 2024, 2025 (24 meses)
  Error promedio: 1.06%

NORMALIZACIÓN:
  num_noticias → potencia 0.75 si excede máximo histórico
  Razón: Explosión de cobertura mediática 2024-2025
  
NOWCASTING 2026:
  Método: Predicción mes por mes con mismo modelo validado
  Adaptable: Funciona con 1-12 meses disponibles
  Actualización: Automática al ejecutar notebook

NIVEL DE CONFIANZA:
  1-2 meses:   Muy Baja (±15%)
  3-4 meses:   Baja (±12%)
  5-6 meses:   Media (±8%)
  7-9 meses:   Alta (±5%)
  10-12 meses: Muy Alta (±3%)

VENTAJA PRINCIPAL:
  Frecuencia mensual vs anual del BCE
  Alerta temprana de cambios económicos
  Basado en señales actuales, no solo historia

